## Why carbon nanotubes?
<div style="text-align: center;"> <img src="images/cnt_vis.jpg" alt="CNT Visualization" style="width: 300px; display: inline-block;"> </div>

Carbon nanotubes (CNTs) are graphene sheets conceptually rolled into seamless cylinders. Their high aspect ratio, low density, mechanical stiffness, and thermal and electrical transport properties make them attractive as nanoscale fillers in polymer nanocomposites. In molecular simulations, CNTs are also useful model systems for studying confinement, interfacial adhesion, functionalization, and load transfer between a polymer matrix and a reinforcing nanostructure.

A CNT is identified by its chiral indices $(n,m)$. These integers define the chiral vector

$$
\mathbf{C}_h=n\mathbf{a}_1+m\mathbf{a}_2,
$$

where $\mathbf{a}_1$ and $\mathbf{a}_2$ are the primitive lattice vectors of graphene. When the graphene sheet is wrapped so that the two ends of $\mathbf{C}_h$ meet, the vector sets the tube circumference and therefore its diameter; its direction sets the chiral angle and atomic arrangement around the tube. Special cases are armchair CNTs $(n,n)$, zigzag CNTs $(n,0)$, and general chiral CNTs $(n,m)$ with $0<m<n$. The indices also strongly influence the CNT's electronic structure.

<div style="text-align: center;"> <img src="images/cnt_chiral.png" alt="Graphene lattice showing the CNT chiral vector" style="width: 300px; display: inline-block;"> </div>

# Carbon Nanotube Simulation Workflow

In this notebook, you will build, simulate, visualize, and analyse CNT systems with molecular dynamics.

The workflow starts by designing a CNT structure from its chiral vector $(N, M)$, choosing the number of periodic repeats, and optionally adding chemical functional groups such as H, OH, NH2, or COOH. The generated CNT structure is then written as a LAMMPS data file and visualized directly in the notebook.

After structure generation, the CNT is relaxed with LAMMPS. The simulation output is analysed with Python by reading thermodynamic data, plotting energies and stresses, and inspecting the trajectory. In the next step, a tensile deformation simulation is prepared to calculate a stress-strain curve and investigate the mechanical response of the CNT.

By the end of this notebook, you should be able to:

- generate pristine and functionalized CNT structures
- visualize CNT structures and trajectories
- run LAMMPS relaxation and deformation simulations
- read and analyse thermodynamic output files
- calculate stress-strain behaviour

Work through the notebook step by step and inspect your results carefully after each simulation stage.

In [ ]:
# Import all necessary libraries

import os
import sys
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend, savgol_filter

from scripts.utils import show_py3dmol
from scripts.utils import show_smiles
from scripts.utils import write_vmd_polymer
from scripts.utils import gen_cnt
from scripts.get_vdw_radius import get_molecular_surface_area

import MDAnalysis as mda

from scripts.run_lammps import run_lammps_background
from scripts.init_lammps import init_lammps
LAMMPS_EXE = init_lammps()

HOME_DIR = os.getcwd()

cols = [
    "step", "temp", "etotal", "ke", "pe", "press",
    "vol", "density", "lx", "ly", "lz",
    "pxx", "pyy", "pzz",
]

### Note about notebook saving warnings

During trajectory visualization, the notebook may sometimes show a warning that it could not be saved. This can happen when large trajectory outputs or interactive viewers are stored in the notebook.

You can usually ignore this warning. If it becomes annoying or prevents saving, click on the affected output cell and clear the output.


## Design your carbon nanotube structure

In this step, you define the CNT that will be generated for the simulation. The nanotube geometry is controlled by the chiral indices $(N, M)$, which define the CNT chirality and diameter. The `repeat` parameter controls how many periodic CNT unit cells are repeated along the tube axis.

You can also choose whether the CNT should be pristine or functionalized. Supported functional groups are:

- `"none"` for a pristine CNT
- `"H"` for hydrogen functionalization
- `"OH"` for hydroxyl functionalization
- `"NH2"` for amine functionalization
- `"COOH"` for carboxyl functionalization

The variable `perc` defines the percentage of CNT carbon atoms that will be functionalized. For example, `perc = 5` means that approximately 5% of the CNT carbon atoms will receive the selected functional group.

Choose these parameters carefully, because they define the structure, system size, periodic length, and chemical composition of the simulation model. For this exercise, the generated CNT structure should not contain more than 2000 atoms, should have at least a height of 15 Anström and minimum radius of 4 Anström.

In [ ]:
 # define the geometry for your cnt structure by the chiral vector
N = 10
M = 1
repeat = 2 # (int)

# define functionalization: "none", "H", "OH", "NH2", "COOH"
functionalization = "OH"

# define percentage of functionalization
perc = 5 


# this is your output name and folder
outname = f"cnt_N{N}_M{M}_repeat{repeat}_perc{perc}{functionalization}"

# use this command to create your cnt and assing you datafile to variable
data_file = gen_cnt(
    N=N,
    M=M,
    repeat=repeat,
    functionalization=functionalization,
    perc=perc,
    outname=outname,
    HOME_DIR=HOME_DIR,
    vacuum=20.0,
    seed=1234,
)

print(data_file)


## Visualize the generated CNT structure

After generating the CNT structure, you can visualize it directly in the notebook using `show_py3dmol()`.

The required input is the LAMMPS data file returned by `gen_cnt()`.

```python
show_py3dmol(data_file)
```

In [ ]:
# visualize your data_file

## Determine the CNT size and surface area

Before running or analysing the simulation, determine the basic geometric properties of your generated CNT. Use the LAMMPS data file `data_file` and read the atom coordinates with `MDAnalysis`.

Your task is to calculate:

- the approximate CNT radius
- the CNT diameter
- the axial length of the CNT
- the lateral surface area of the CNT

Use the atom coordinates to estimate the centre of the tube in the `xy` plane. Then calculate the radial distance of each atom from this centre. The average radial distance gives an estimate of the CNT radius.

The CNT surface area can be approximated as the surface area of a cylinder:

$$
A = \pi d L
$$

where \(d\) is the CNT diameter and \(L\) is the CNT length along the tube axis.

Helpful code structure:

```python
# read the LAMMPS data file into MDAnalysis atoms object
u = mda.Universe(
    your_data_file,
    format="DATA",
    atom_style="id type x y z",
)

# get atomic coordinates
u.atoms.positions

# separate x, y, z coordinates
x = coords[:, 0]
y = ..
..

# calculate the centre of the CNT in the xy plane
x0 = ...
y0 = ...

# calculate radial distances from the CNT axis
r = np.sqrt((x - x0)**2 + (y - y0)**2)

# estimate radius and diameter
radius = ...
diameter ..
pi = np.pi

# estimate CNT height from zmin and zmax and cylindrical surface area

height = ...
area = ...

print("area     =", area)
```

In [ ]:
u = mda.Universe(
    data_file,
    format="DATA",
    atom_style="id type x y z",
)

coords = u.atoms.positions

# separate x, y, z coordinates
x = coords[:, 0]
y = coords[:, 1]
z = coords[:, 2]

# calculate the centre of the CNT in the xy plane
x0 = x.mean()
y0 = y.mean()

# calculate radial distances from the CNT axis
r = np.sqrt((x - x0)**2 + (y - y0)**2)
radius = r.mean()

diameter = 2*radius
pi = np.pi

# estimate CNT height from zmin and zmax and cylindrical surface area

height = z.max()-z.min()

area = diameter * pi * height

# Units of length in Angström

print("radius   =", radius)
print("height   =", height)
print("area     =", area)



## Calculate the molecular surface area

In this step, calculate the approximate molecular surface area of the generated CNT using `get_molecular_surface_area()`.

The function reads the atom coordinates from the LAMMPS data file and assigns each atom a van der Waals radius based on its element. With `probe_radius=0.0`, the calculated area corresponds to an approximate van der Waals surface area of the atomistic CNT structure.

This value does not require choosing a CNT wall thickness. Instead, the surface is constructed from the atoms themselves. Therefore, it is useful for comparing different CNT structures or functionalizations, but it is not the same as the smooth cylindrical surface area.

```python
get_molecular_surface_area(data_file, probe_radius)


In [ ]:
vdw_surface = get_molecular_surface_area(data_file, probe_radius=0.0)

print(vdw_surface)


The cylindrical surface area and the vdW molecular surface area are not expected to be identical. The cylindrical area uses a smooth geometric approximation based on the CNT radius and length. In contrast, the vdW surface area is calculated from overlapping atomic spheres using atomic van der Waals radii.

Therefore, the vdW surface is usually larger. It may include accessible contributions from both the outer and inner side of the CNT wall, and it also reflects the atomistic roughness of the structure and any functional groups present.

## Run the CNT relaxation simulation

In this step, LAMMPS runs the relaxation simulation defined in `in.relaxation`.

The simulation first reads the generated CNT structure and force-field settings. Then it performs an energy minimization where the atomic positions are relaxed. During this minimization, the simulation box is allowed to relax only in the CNT axial direction, \(z\), using `fix box/relax`. This can reduce artificial axial stress in the initial periodic CNT structure.

After the minimization, the system is equilibrated at 300 K using an NVT simulation. During this part, the box size is fixed and only the atoms move. The trajectory is written to `relaxation.lammpstrj`, thermodynamic quantities are written to `relaxation.dat`, and the final relaxed structure is written to `relaxation.data`.

The calculation is launched with `mpirun` using one MPI process. For this small CNT system, one process is usually sufficient.

In [ ]:
input_file = "in.relaxation" 

cmd = ["mpirun", "-np", "2", "--oversubscribe", "--bind-to", "none", LAMMPS_EXE, "-in", input_file]
process, stdout_thread, stderr_thread = run_lammps_background(cmd, outname)

## Visualize the relaxation trajectory

After the LAMMPS relaxation has finished, you can visualize the trajectory file with `show_py3dmol()`.

The function needs the original LAMMPS data file as topology input and the LAMMPS trajectory file as coordinate input. The trajectory file `relaxation.lammpstrj` is written during the simulation. `Path` is used to build file paths in a platform-independent way. The `/` operator joins the output folder with the trajectory filename.

```python
traj_file = Path(dir) / "relaxation.lammpstrj"

show_py3dmol(data_file, traj_file, frame_stride=10)
```

In [ ]:
data_file = Path(outname) / "relaxation.data"
traj_file = Path(outname) / "relaxation.lammpstrj"

show_py3dmol(data_file, traj_file, frame_stride=10)

## Read the thermodynamic output

After the simulation, LAMMPS writes selected thermodynamic quantities to `relaxation.dat`. Read this file with `pandas` so that you can inspect and analyse the simulation results.

The file contains whitespace-separated columns, and lines starting with `#` are comments. Define the column names first, then read the file into a dataframe.

```python

df = pd.read_csv(
    thermo_file,
    sep=r"\s+",
    comment="#",
    names=cols,
)
```
Use `df.head()` to inspect the first rows of the dataframe and identify the available quantities:

```python
df.head()
```

A dataframe column can be accessed by its name:

```python
df["temp"]
df["etotal"]
```

Rows and columns can also be selected by their numerical position using `iloc`:

```python
df.iloc[:10]       # first 10 rows
df.iloc[:, 0]      # first column
df.iloc[:10, 1:4]  # first 10 rows and columns 2 to 4
```

In [ ]:
thermo_file = Path(outname) / "relaxation.dat"

df = pd.read_csv(
    thermo_file,
    sep=r"\s+",
    comment="#",
    names=cols,
)

df.head()

## Plot simulation quantities

After reading the thermodynamic output into a pandas dataframe, you can plot any column against another column using `matplotlib`.

For this simulation, energies are reported in eV. The LAMMPS timestep is 0.5 fs.

Plot the total energy against time in ps.

```python
plt.plot(x_values, y_values) 
plt.xlabel("label for x-axis") 
plt.ylabel("label for y-axis") 
plt.title("plot title") 
plt.show()
```

In [ ]:
time = df['step'] / 2000  # time in ps
etotal = df['etotal']     # energy in eV

plt.plot(time, etotal) 
# plt.xlabel("label for x-axis") 
# plt.ylabel("label for y-axis") 
# plt.title("plot title") 
plt.show()

## Prepare a tensile deformation simulation

After the relaxation simulation has finished, use the relaxed CNT structure as the starting point for the tensile test.

The provided `in.deform` file is already configured correctly for the tensile-deformation simulation. Open the file and examine the differences relative to the relaxation input script.

The main differences are:

1. **The relaxed structure is used as input.**

   The `read_data` command reads the output data file generated by the relaxation simulation rather than the original CNT structure. This allows the tensile test to start from the relaxed configuration.

2. **The minimization section has been removed.**

   The box-relaxation and energy-minimization commands are no longer required because the structure has already been relaxed. Therefore, the following section is not included in `in.deform`:

   ```lammps
   fix axial all box/relax z 0.0 vmax 0.001

   min_style cg
   minimize 1.0e-8 1.0e-10 1000 10000

   unfix axial
   reset_timestep 0
   ```

3. **No new velocities are created.**

   The following command from the relaxation input script is not included:

   ```lammps
   velocity all create 300.0 12345 mom yes rot yes dist gaussian
   ```

   The tensile simulation should continue using the atomic velocities stored in the relaxed output data file. Make sure that the relaxation simulation writes the velocities to this file and that `in.deform` reads the correct file.


4. **Separate output filenames are used.**

   The output filenames are different from those used during relaxation. This prevents the tensile-deformation simulation from overwriting the relaxation results.

5. **Tensile deformation is applied along the CNT axis.**

   The simulation box is continuously stretched along the axial \(z\)-direction using:

   ```lammps
   fix deform all deform 1 z vel 5.0e-3 units box remap x
   ```

   More generally, the deformation velocity can be specified as:

   ```lammps
   fix deform all deform 1 z vel <V> units box remap x
   ```

   Here, `V` is the rate of change of the box length and has units of distance per time. With `units metal`, the deformation velocity is expressed in Å/ps.

   A reasonable starting range is:

   ```text
   1.0e-3 to 1.0e-1 Å/ps
   ```

Take care when selecting the deformation velocity. A very high value may stretch the CNT too rapidly and produce a strongly rate-dependent, nonequilibrium response. A lower value gives the structure more time to respond and relax during deformation, but it also requires a longer simulation.

During the tensile test, the system is maintained at constant temperature using NVT while the simulation box is stretched along the CNT axis.

In [ ]:
input_file = "in.deform"

cmd = ["mpirun", "-np", "2", "--oversubscribe", "--bind-to", "none", LAMMPS_EXE, "-in", input_file]
process, stdout_thread, stderr_thread = run_lammps_background(cmd, outname)

In [ ]:
data_file = Path(outname) / "relaxation.data"
traj_file = Path(outname) / "deform.lammpstrj"

show_py3dmol(data_file, traj_file, frame_stride=1)

## Read the deformation output

After the tensile deformation simulation has finished, read the thermodynamic output file into a pandas dataframe.

Make sure that the filename matches the name you used in `in.deform`. For example, if your LAMMPS input contains:

    variable print_file string deform.dat

then the output file is called `deform.dat`.

Read the dataframe!

In [ ]:
deform_file = Path(outname) / "deform.dat"

df = pd.read_csv(
    deform_file,
    sep=r"\s+",
    comment="#",
    names=cols,
)


## Plot stress against strain

For the tensile deformation simulation, analyse how the axial stress changes as the carbon nanotube is stretched along the $z$-direction.

The engineering strain, $\varepsilon$, must be calculated from the simulation box length in the tensile direction:

$$
\varepsilon = \frac{L_z-L_{z_0}}{L_{z_0}}
$$

where:

- $L_z$ is the current box length, stored in the `lz` column.
- $L_{z_0}$ is the initial box length, given by the first value in the `lz` column.

The axial stress in the tensile direction is stored in the `pzz` column. The `pzz` values are given in **bar**, so they must be converted to **MPa** before plotting:

$$
1\ \mathrm{bar} = 0.1\ \mathrm{MPa}
$$

Calculate the strain and convert the axial stress to MPa. Then plot the axial stress against strain by completing the code below:
```python
    strain = ...

    plt.plot(strain,..
```

The axial stress in the tensile direction is stored in the `pzz` column in MPa.

In [ ]:
strain = (df['lz']-df['lz'].loc[0])/df['lz']
stress = df['pzz']*0.1

plt.plot(strain, stress)
plt.show

## Correct the axial stress using the CNT cross-sectional area

The stress printed by LAMMPS is based on the full simulation box volume. For an isolated CNT, the box contains a large amount of vacuum in the `x` and `y` directions. Therefore, the printed `pzz` value does not directly represent the true axial stress of the CNT wall.

To correct the stress, rescale it from the simulation box cross-sectional area to the effective CNT cross-sectional area.

The effective cross-sectional area of a single-walled CNT can be approximated as:

$$
A_\mathrm{CNT} = 2 \pi R t
$$

where:

- $R$ is the CNT radius
- $t$ is the effective CNT wall thickness
- a common choice is $t = 3.4~\text{Å}$, corresponding to the graphite interlayer spacing

The simulation box cross-sectional area is:

$$
A_\mathrm{box} = L_x L_y
$$

where $L_x$ and $L_y$ are the box lengths in the directions perpendicular to the CNT axis.

The corrected axial CNT stress is then:

$$
\sigma_{zz,\mathrm{CNT}} =
p_{zz}
\frac{A_\mathrm{box}}{A_\mathrm{CNT}}
$$

Your task is to calculate $A_\mathrm{CNT}$, calculate $A_\mathrm{box}$ from the dataframe columns, and then create a corrected stress column.

Helpful code structure:

    R = ...
    t = ...

    u = mda.Universe(...)
    Lx, Ly, Lz = u.dimensions[:3]
    
    A_cnt = ...
    A_box = ...

    df["pzz_corrected"] = ...

Then plot the corrected stress against the axial strain:
